# Scratch: try out preprocessing and models

A place to experiment. Not committed (see `.gitignore`), so break things freely.

When something works, move the clean version into `src/credit_risk/models.py`
and check it with `pytest tests/test_models.py`.

**Workflow**

1. Sections 1 and 2 show what the preprocessor and a single fit do.
2. Section 3 sets up cross-validation once: shared folds, metrics and two helpers.
3. Section 4 lists the models in `models = {...}` and runs them all.
4. Section 5 shows the comparison as a coloured table, then as a chart.

In [1]:
# Reload src/ code on every run, so edits to models.py show up without restarting.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from credit_risk.data import TARGET, load_split
from credit_risk.preprocess import build_preprocessor

train = load_split("train")
X_train, y_train = train.drop(columns=TARGET), train[TARGET]
X_train.shape, y_train.mean()

((24000, 24), np.float64(0.22120833333333334))

## 1. What does the preprocessor do?

In [3]:
pre = build_preprocessor()
Xt = pre.fit_transform(X_train)
print(X_train.shape, "->", Xt.shape)
pre.get_feature_names_out()[:15]

(24000, 24) -> (24000, 28)


array(['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
       'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2',
       'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'SEX_2'],
      dtype=object)

## 2. Fit a model by hand

One fit on all of train, scored on the same rows. Useful to see what `fit` and
`predict_proba` return, but the score is optimistic. Section 3 does it properly.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Before training, weights are placeholders; each iteration of coefficients calculates a loss,
# and directs each weight to lower it; nudge in that direction;
# the goal is to reach convergence where the loss no longer improves
m = LogisticRegression(max_iter=1000).fit(Xt, y_train)
proba = m.predict_proba(Xt)[:, 1]
print(proba[:5])
print("train AUC:", roc_auc_score(y_train, proba))  # on training data, so optimistic

[0.50744388 0.14212182 0.2423766  0.13209962 0.23678692]
train AUC: 0.7250465202142095


## 3. Cross-validation setup

Run this section once. It defines:

- `cv`: one set of 5 stratified folds that every model is scored on.
- `scoring`: the metrics from the M2 plan.
- `run_cv(name, model)`: cross-validates one model and stores its scores in `results`.
- `compare()`: a coloured table of every model in `results`, side by side.

In [ ]:
import warnings

import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline

from credit_risk.config import default_config_path, load_config

# The dummy never predicts "default", so precision is 0/0 for it. Expected; hide the warning.
warnings.filterwarnings("ignore", message="Precision is ill-defined")

cfg = load_config(default_config_path())

# One set of folds shared by every model, so score differences come from the
# model, not from which rows landed in which fold. Stratified: each fold keeps
# the ~22% default rate.
cv = StratifiedKFold(n_splits=cfg.cv_folds, shuffle=True, random_state=cfg.seed)

# Metrics from the M2 plan. Accuracy left out: predicting "no default" for
# everyone already scores 0.78. average_precision is PR-AUC.
scoring = ["roc_auc", "average_precision", "recall", "precision", "f1"]

results = {}  # model name -> its summary table; filled by run_cv


def run_cv(name, model):
    """Cross-validate `model` behind the preprocessor on the shared folds;
    store and show its scores."""
    # Preprocessor inside the pipeline: it is refit on each training fold,
    # so the held-out fold never influences the scaling.
    pipe = make_pipeline(build_preprocessor(), model)
    scores = pd.DataFrame(
        cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    )  # one row per fold

    # One row per metric. val = the held-out fold; train = the folds it was fit on.
    summary = pd.DataFrame(
        {
            "val_mean": [scores[f"test_{m}"].mean() for m in scoring],
            "val_std": [scores[f"test_{m}"].std() for m in scoring],  # spread across folds
            "train_mean": [scores[f"train_{m}"].mean() for m in scoring],
        },
        index=scoring,
    )
    summary["gap"] = summary["train_mean"] - summary["val_mean"]  # large = overfitting

    results[name] = summary
    print(f"{name}: {scores['fit_time'].mean():.2f}s to fit per fold")
    return summary.round(3)


def _shade(score):
    """Blue cell background, darker for higher scores (scores are on a 0-1 scale)."""
    return f"background-color: rgba(49, 130, 189, {0.75 * score:.2f})"


def compare():
    """Mean validation score of every model run so far. Rows = metrics, columns = models."""
    table = pd.DataFrame({name: s["val_mean"] for name, s in results.items()})
    return (
        table.style.format("{:.3f}")
        .map(_shade)  # lab4-style colouring, without needing matplotlib
        .highlight_max(axis=1, props="font-weight: bold")  # best model per metric
    )

## 4. Models

Add or remove entries in `models`, then run the cell. Each one is
cross-validated on the same folds and stored in `results` under its name.
Use a distinct name per setting, for example `"logreg_C0.1"`.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "dummy": DummyClassifier(strategy="prior"),  # floor: ignores the features
    "logreg": LogisticRegression(max_iter=1000),  # first real model
    "forest": RandomForestClassifier(random_state=cfg.seed, class_weight="balanced"),
    "lgbm": LGBMClassifier(verbose=-1, random_state=cfg.seed),
    "xgboost": XGBClassifier(),
}

results = {}  # start fresh, so removed models drop out of the comparison
for name, model in models.items():
    run_cv(name, model)

dummy: 0.02s to fit per fold
logreg: 0.05s to fit per fold
forest: 3.63s to fit per fold
lgbm: 1.16s to fit per fold
xgboost: 0.45s to fit per fold


## 5. Compare

### 5.1 Table

Mean validation score per metric (rows) and model (columns). Darker = higher;
bold = best on that row.

For one model's detail, including its fold-to-fold spread and train/validation
gap, look at `results["logreg"]`:

- `val_std`: how much the score moves between folds. Small = stable.
- `gap`: train minus validation. Near 0 = generalizes; large = overfitting.

In [7]:
compare()

,dummy,logreg,forest,lgbm,xgboost
roc_auc,0.500,0.722,0.763,0.776,0.760
average_precision,0.221,0.503,0.529,0.555,0.523
recall,0.000,0.236,0.481,0.361,0.367
precision,0.000,0.720,0.558,0.667,0.629
f1,0.000,0.356,0.517,0.468,0.464


### 5.2 Chart

Grouped bars: metrics on the x-axis, one colour per model. Hover a bar for its
numbers; the black line is ±1 std across the 5 folds.

In [8]:
import altair as alt

# Long format: one row per (model, metric)
df = pd.concat(results, names=["model", "metric"]).reset_index()
df["low"], df["high"] = df["val_mean"] - df["val_std"], df["val_mean"] + df["val_std"]

base = alt.Chart(df).encode(
    x=alt.X("metric:N", sort=scoring, title=None, axis=alt.Axis(labelAngle=-45)),
    xOffset="model:N",  # places models side by side within each metric
    color=alt.Color("model:N", title="Model"),
)
bars = base.mark_bar().encode(
    y=alt.Y(
        "val_mean:Q", scale=alt.Scale(domain=[0, 1]), title="Validation score (mean of 5 folds)"
    ),
    tooltip=[
        "model:N",
        "metric:N",  # hover shows these
        alt.Tooltip("val_mean:Q", format=".3f"),
        alt.Tooltip("val_std:Q", format=".3f"),
    ],
)
spread = base.mark_rule(color="black").encode(y="low:Q", y2="high:Q")  # ±1 std across folds

(bars + spread).properties(width=450, height=280).configure_axis(
    labelFontSize=13, titleFontSize=14
).configure_legend(labelFontSize=13, titleFontSize=14)

alt.LayerChart(...)

## 6. Try your `get_model`

Raises until you implement it in `src/credit_risk/models.py`. Save the file and
re-run this cell. Once it works, `models` in section 4 can use `get_model("logreg", seed=cfg.seed)`
in place of the hand-written models.

In [ ]:
from credit_risk.models import get_model

for name in ["dummy", "logreg", "lgbm"]:
    print(name, get_model(name, seed=0))